In [ ]:
# these are the libraries that you will need throughout the assignment
import numpy as np 
import pandas as pd

import matplotlib.pyplot as plt 
import seaborn as sns
%matplotlib inline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import MDS

from matplotlib.colors import ListedColormap


RSEED = 8

In [ ]:
df1 = pd.DataFrame() # change this
df1 = pd.read_csv("df1.csv", low_memory=False)

#header=1 bc we omit row 0, column names start at row=1

"""df1 = df1.drop(columns=["CONCOHORT_LABEL", 
                        "CONCOHORT_lbl", 
                        "pt_color", 
                        "is_outlier"])"""

df1

In [ ]:
cohort_map = {1: "PD", 2: "HC", 4: "Prodromal PD"}
sex_map    = {0: "Female", 1: "Male"}

# Build a contingency table (counts) by cohort x sex
ct = (
    df1
    .loc[df1["CONCOHORT"].isin(cohort_map.keys()), ["CONCOHORT", "SEX"]]
    .assign(
        CONCOHORT_lbl=lambda d: d["CONCOHORT"].map(cohort_map),
        SEX_lbl=lambda d: d["SEX"].map(sex_map),
    )
    .groupby(["CONCOHORT_lbl", "SEX_lbl"])
    .size()
    .unstack(fill_value=0)
    .reindex(["PD", "Prodromal PD", "HC"])  # desired x-axis order
)

ax = ct.plot(kind="bar", stacked=True, color=["pink", "lightblue"])
plt.title("Patients per Cohort (Sex)")
plt.xlabel("Cohort")
plt.ylabel("Number of patients")
plt.xticks(rotation=0)
plt.legend(title="Sex")
plt.tight_layout()
plt.show()


In [ ]:
df1["subgroup"].value_counts().plot(kind="bar", color = ['lightblue'])
plt.title("Subgroup sizes")
plt.ylabel("Number of subjects")
plt.tight_layout()
plt.show()

In [ ]:
cohort_map = {1: "PD", 2: "HC", 4: "Prodromal"}

df1_plot = df1.copy()
df1_plot["CONCOHORT_LABEL"] = df1_plot["CONCOHORT"].map(cohort_map)

# keep only cohorts we care about (and implicitly exclude cohort 3 / unmapped)
df1_plot = df1_plot[df1_plot["CONCOHORT_LABEL"].notna()].copy()


# order subgroups by frequency (largest first)
order = df1_plot["subgroup"].value_counts().index.tolist()

plt.figure(figsize=(8, 6))

sns.countplot(
    data=df1_plot,
    x="subgroup",
    hue="CONCOHORT_LABEL",
    palette="Set3",
    order=order
)

plt.title("Subgroup sizes by Concohort")
plt.ylabel("Number of subjects")
plt.xlabel("Subgroup")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Cohort", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

pd.crosstab(df1_plot["subgroup"], df1_plot["CONCOHORT_LABEL"])


In [ ]:
#missing values for: clinical severity, cognitive/neuropsychological measures
cols_to_check = [
    "updrs1_score", "updrs2_score", "updrs3_score", "updrs3_score_on", "updrs4_score",
    "updrs_totscore", "updrs_totscore_on",
    "LEDD", "age_DATSCAN", "moca", "MCI_testscores", 
    "SDMTOTAL", "TMT_A", "TMT_B", "VLTANIM", "rem",
    "upsit", "gds", "PRIMDIAG", "educ", "EDUCYRS", "Field Strength"

]

missing_values = df1[cols_to_check].isna().sum()

print(missing_values)


missing_pct = df1[cols_to_check].isna().mean() * 100

plt.figure()
missing_pct.plot(kind="bar", color = ['pink'])
plt.ylabel("Percentage missing (%)")
plt.title("Percentage of missing values per Variable (clinical severity, cognitive/neuropsycological measures)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# --- per subgroup ---
miss_by_subgroup_count = df1.groupby("subgroup")[cols_to_check].apply(lambda g: g.isna().sum())
miss_by_subgroup_pct = df1.groupby("subgroup")[cols_to_check].apply(lambda g: g.isna().mean().mul(100))

# (optional) sort subgroups by size
subgroup_order = df1["subgroup"].value_counts().index
miss_by_subgroup_count = miss_by_subgroup_count.reindex(subgroup_order)
miss_by_subgroup_pct = miss_by_subgroup_pct.reindex(subgroup_order)

display(miss_by_subgroup_count)     # counts missing per subgroup
display(miss_by_subgroup_pct.round(1))  # % missing per subgroup


In [ ]:
plt.figure(figsize=(12, max(4, 0.35 * miss_by_subgroup_pct.shape[0])))

plt.imshow(miss_by_subgroup_pct.values, aspect="auto")
plt.colorbar(label="Missing (%)")

plt.xticks(range(len(cols_to_check)), cols_to_check, rotation=45, ha="right")
plt.yticks(range(len(miss_by_subgroup_pct.index)), miss_by_subgroup_pct.index)

plt.title("Missing values per subgroup (%): clinical severity + cognitive/neuropsych measures")
plt.tight_layout()
plt.show()


In [ ]:
cohort_map = {1: "PD", 2: "HC", 4: "Prodromal"}

df_plot = df1.copy()
df_plot["CONCOHORT_lbl"] = df_plot["CONCOHORT"].map(cohort_map)

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 5))

sns.scatterplot(
    data=df_plot,
    x="updrs3_score",
    y="updrs1_score",
    hue="CONCOHORT_lbl",
    alpha=0.4
)

plt.title("Motor vs Non-motor symptoms")
plt.xlabel("UPDRS III (Motor)")
plt.ylabel("UPDRS I (Non-motor)")
plt.tight_layout()
plt.show()



### What the plot shows:
- PD spreads widely on UPDRS III (as expected)
- Prodromal + HC cluster at low motor scores
- Non-motor symptoms (UPDRS I) overlap more across groups 
- Slight positive association between motor and non-motor severity

In [ ]:
df1.shape

In [ ]:
cohort_map = {1: "PD", 2: "HC", 4: "Prodromal"}

df_plot1 = df1.copy()
df_plot1["CONCOHORT_lbl"] = df_plot1["CONCOHORT"].map(cohort_map)

df_pd = df_plot1[df_plot1["CONCOHORT_lbl"] == "PD"].copy()

import matplotlib.pyplot as plt
import numpy as np

x = df_pd["age"]
y = df_pd["agediag"]

plt.figure()
plt.scatter(x, y)

# Add 45-degree reference line
max_val = max(x.max(), y.max())
plt.plot([0, max_val], [0, max_val])

plt.xlabel("Current Age")
plt.ylabel("Age at Diagnosis")
plt.title("Age vs Age at Diagnosis (PD Cohort)")
plt.show()

df_pd["disease_duration"] = x - y
df_pd["disease_duration"].describe()



In [ ]:
cohort_map = {1: "PD", 2: "HC", 4: "Prodromal"}

df_plot1 = df1.copy()
df_plot1["CONCOHORT_lbl"] = df_plot1["CONCOHORT"].map(cohort_map)

df_pd = df_plot1[df_plot1["CONCOHORT_lbl"] == "PD"].copy()

import matplotlib.pyplot as plt
import numpy as np

# --- columns ---
x = df_pd["age"]
y = df_pd["agediag"]
sex = df_pd["SEX"]  # change this if your sex column has a different name

# Keep only complete rows (x, y, sex)
mask = x.notna() & y.notna() & sex.notna()
x = x[mask]
y = y[mask]
sex = sex[mask]

# ---- Scatter, color-coded by sex ----
plt.figure()

# Plot each sex as a separate scatter so we get a legend
for s in sorted(sex.unique()):
    idx = (sex == s)
    plt.scatter(x[idx], y[idx], label=str(s), alpha=0.8)

# 45-degree reference line (Age = Age at Dx)
max_val = max(x.max(), y.max())
plt.plot([0, max_val], [0, max_val])

# ---- Regression line (overall, across PD cohort) ----
m, b = np.polyfit(x.astype(float), y.astype(float), 1)
xx = np.linspace(0, max_val, 200)
plt.plot(xx, m * xx + b, linewidth=2, label=f"Regression (y={m:.2f}x+{b:.2f})")

plt.xlabel("Current Age")
plt.ylabel("Age at Diagnosis")
plt.title("Age vs Age at Diagnosis (PD Cohort), colored by sex")
plt.legend(title="Sex")
plt.show()

# ---- Disease duration summary ----
df_pd = df_pd.loc[mask].copy()
df_pd["disease_duration"] = df_pd["age"] - df_pd["agediag"]
df_pd["disease_duration"].describe()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# 1. Get subcortical variables
# -----------------------------
all_cols = df1.columns.tolist()

subcortical_cols = [
    col for col in all_cols
    if (
        col.startswith("Left-")
        or col.startswith("Right-")
        or col in [
            "3rd-Ventricle",
            "4th-Ventricle",
            "5th-Ventricle",
            "Brain-Stem",
            "CSF",
            "WM-hypointensities",
            "Left-WM-hypointensities",
            "Right-WM-hypointensities",
            "non-WM-hypointensities",
            "Left-non-WM-hypointensities",
            "Right-non-WM-hypointensities",
            "Optic-Chiasm",
            "CC_Posterior",
            "CC_Mid_Posterior",
            "CC_Central",
            "CC_Mid_Anterior",
            "CC_Anterior"
        ]
    )
]

# Remove duplicates (just in case)
subcortical_cols = list(dict.fromkeys(subcortical_cols))

print("Number of subcortical variables:", len(subcortical_cols))

# -----------------------------
# 2. Treat 0.0 as missing
# -----------------------------
subcortical_data = df1[subcortical_cols].replace(0, np.nan)

# -----------------------------
# 3. Overall missing values
# -----------------------------
missing_values = subcortical_data.isna().sum()
print("\nMissing values per variable:")
print(missing_values)

missing_pct = subcortical_data.isna().mean() * 100

# Plot
plt.figure(figsize=(14, 6))
missing_pct.plot(kind="bar", color="pink")
plt.ylabel("Percentage missing (%)")
plt.title("Missing values per subcortical variable (NaN + 0 treated as missing)")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# -----------------------------
# 4. Missing per subgroup
# -----------------------------
miss_by_subgroup_count = df1.groupby("subgroup")[subcortical_cols].apply(
    lambda g: g.replace(0, np.nan).isna().sum()
)

miss_by_subgroup_pct = df1.groupby("subgroup")[subcortical_cols].apply(
    lambda g: g.replace(0, np.nan).isna().mean() * 100
)

# Sort subgroups by size
subgroup_order = df1["subgroup"].value_counts().index
miss_by_subgroup_count = miss_by_subgroup_count.reindex(subgroup_order)
miss_by_subgroup_pct = miss_by_subgroup_pct.reindex(subgroup_order)

print("\nMissing counts by subgroup:")
display(miss_by_subgroup_count)

print("\nMissing % by subgroup:")
display(miss_by_subgroup_pct.round(1))

In [ ]:
(df1[subcortical_cols] == 0).sum().sort_values(ascending=False).head(10)